# MASI Full Dataset Pipeline

This notebook runs the proposal-aligned MASI pipeline for `configs/Full_dataset.json`.

It covers the complete prepared `full_dataset` Kaggle flow:

1. clone MASI from GitHub into `/kaggle/working/MASI`,
2. validate the prepared dataset at `/kaggle/input/datasets/dheerajrajanala/masi-amazon-csj-full-dataset`,
3. reuse the dataset's existing `images/` folder and manifests,
4. run Phase 1 behavior alignment, Phase 2 dual RQ-VAE tokenization, and Phase 3 recommendation training,
5. inspect summaries and package run outputs.

The attached dataset is expected to contain `Clothing_Shoes_and_Jewelry.jsonl`, `meta_Clothing_Shoes_and_Jewelry.jsonl`, `images/`, `image_download_manifest.json`, and `subset_manifest.json`. This notebook does not rebuild the prepared subset or redownload images on Kaggle.

The Kaggle path intentionally derives a bounded runtime config from `configs/Full_dataset.json`. The default profile is now `long_safe`, a larger continuation run intended to improve metrics while staying inside the current in-memory CLIP embedding limit. Use `scaled_safe` for the previously completed smaller profile, or `smoke_safe` for a quick pipeline check.

The notebook now auto-restores a previously exported run bundle from `/kaggle/input`, advances to the next bounded user-rank data chunk, continues training from the newest restored checkpoints, and exports a refreshed bundle with final checkpoints, retained periodic checkpoints, resolved configs, summaries, manifests, fused IDs, and chunk state for the next run. Attach the previous long-safe bundle as the Kaggle Dataset `masi-long-safe-resume-bundle`; the restore step normalizes either a zip, a direct run folder, `outputs/<run_name>`, or `masi_artifacts/outputs/<run_name>` into the writable `/kaggle/working/masi_artifacts/outputs/<run_name>` path expected by the notebook. The CLIP cell writes a standalone local model directory under `/kaggle/working/masi_artifacts/hf_models/` when no attached CLIP model dataset is found.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile


REPO_URL = "https://github.com/pradyunuydarp/MASI.git"
REPO_BRANCH = "main"
KAGGLE_WORKING_ROOT = Path("/kaggle/working")
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
RUNNING_ON_KAGGLE = KAGGLE_WORKING_ROOT.exists() and KAGGLE_INPUT_ROOT.exists()
USE_GIT_CLONE_ON_KAGGLE = True
KAGGLE_REPO_DIR = KAGGLE_WORKING_ROOT / "MASI"
KAGGLE_FULL_DATASET_DIR = KAGGLE_INPUT_ROOT / "datasets" / "dheerajrajanala" / "masi-amazon-csj-full-dataset"
REQUIRED_DATASET_ENTRIES = [
    "Clothing_Shoes_and_Jewelry.jsonl",
    "meta_Clothing_Shoes_and_Jewelry.jsonl",
    "images",
    "image_download_manifest.json",
    "subset_manifest.json",
]


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "masi").exists():
            return candidate
    raise FileNotFoundError("Could not find the MASI repository root from this notebook location.")


if RUNNING_ON_KAGGLE and USE_GIT_CLONE_ON_KAGGLE:
    REPO_DIR = KAGGLE_REPO_DIR
    KAGGLE_WORKING_ROOT.mkdir(parents=True, exist_ok=True)
    os.chdir(KAGGLE_WORKING_ROOT)
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            REPO_BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
        cwd=KAGGLE_WORKING_ROOT,
    )
else:
    REPO_DIR = find_repo_root(Path.cwd())
os.chdir(REPO_DIR)

SOURCE_CONFIG_PATH = REPO_DIR / "configs" / "Full_dataset.json"
CONFIG_PATH = SOURCE_CONFIG_PATH
with SOURCE_CONFIG_PATH.open("r", encoding="utf-8") as handle:
    FULL_DATASET_CONFIG = json.load(handle)

STORAGE_ROOT = KAGGLE_WORKING_ROOT / "masi_artifacts" if RUNNING_ON_KAGGLE else REPO_DIR
RAW_DIR = REPO_DIR / "data" / "raw" / "amazon_reviews_2023"
RAW_REVIEWS_PATH = RAW_DIR / "Clothing_Shoes_and_Jewelry.jsonl"
RAW_METADATA_PATH = RAW_DIR / "meta_Clothing_Shoes_and_Jewelry.jsonl"
PREPARED_DIR = REPO_DIR / "data" / "full_dataset"
RUN_ROOT = STORAGE_ROOT / "outputs" / "amazon_csj_full_dataset_train"

DATASET_CONFIG = FULL_DATASET_CONFIG["dataset"]
REVIEWS_RELPATH = DATASET_CONFIG.get("reviews_relpath") or "Clothing_Shoes_and_Jewelry.jsonl"
METADATA_RELPATH = DATASET_CONFIG.get("metadata_relpath") or "meta_Clothing_Shoes_and_Jewelry.jsonl"
ATTACHED_DATASET_DIR = None
if RUNNING_ON_KAGGLE:
    if all((KAGGLE_FULL_DATASET_DIR / entry).exists() for entry in REQUIRED_DATASET_ENTRIES):
        ATTACHED_DATASET_DIR = KAGGLE_FULL_DATASET_DIR
    for slug in ([] if ATTACHED_DATASET_DIR is not None else DATASET_CONFIG.get("kaggle_input_slugs", [])):
        candidate_paths = [KAGGLE_INPUT_ROOT / str(slug)]
        nested_root = KAGGLE_INPUT_ROOT / "datasets"
        if nested_root.exists():
            candidate_paths.extend(sorted(nested_root.glob(f"*/{slug}")))
        for candidate in candidate_paths:
            if (candidate / REVIEWS_RELPATH).exists() and (candidate / METADATA_RELPATH).exists():
                ATTACHED_DATASET_DIR = candidate
                break
        if ATTACHED_DATASET_DIR is not None:
            break

ACTIVE_PREPARED_DIR = ATTACHED_DATASET_DIR if ATTACHED_DATASET_DIR is not None else PREPARED_DIR
USING_ATTACHED_PREPARED_DATASET = ATTACHED_DATASET_DIR is not None

# Toggle these per runtime. The defaults avoid accidental large downloads and reruns.
RUN_PIP_INSTALL = True
PREFETCH_IMAGES = False
FORCE_PREPARE = False
FORCE_TRAIN = False
CONTINUE_TRAINING_FROM_CHECKPOINTS = True
AUTO_RESTORE_RESUME_BUNDLE = True
EXPORT_BUNDLE = True
IMAGE_WORKERS = 16
KAGGLE_CHECKPOINT_SAVE_STEPS = 25
RESUME_BUNDLE_INPUT_SLUGS = ["masi-long-safe-resume-bundle"]
ADVANCE_DATA_CHUNK_EACH_RUN = True
MANUAL_DATA_CHUNK_INDEX = None
DATA_CHUNK_BY = "user_rank"

# The prepared Kaggle dataset can be large, but this code path currently keeps
# CLIP embeddings in memory. These defaults run a bounded slice from the full
# attached dataset to avoid Kaggle SIGKILL/OOM. Set USE_KAGGLE_SAFE_LIMITS = False
# only on a larger machine or after adding sharded embedding persistence.
#
# Safe profile guidance:
# - smoke_safe: fastest end-to-end integration check.
# - scaled_safe: previously completed ~2 hour metric run.
# - long_safe: default continuation metric-improvement run.
# If long_safe OOMs on Kaggle, fall back to scaled_safe or lower max_items before max_users.
USE_KAGGLE_SAFE_LIMITS = RUNNING_ON_KAGGLE
KAGGLE_SAFE_PROFILE = "long_safe"

KAGGLE_SAFE_PROFILES = {
    "smoke_safe": {
        "max_users": 512,
        "max_items": 1024,
        "max_review_records": 2_000_000,
        "clip_batch_size": 8,
        "train_batch_size": 16,
        "alignment_batch_size": 128,
        "tokenization_batch_size": 128,
        "tokenization_epochs": 4,
        "history_max_tokens": 96,
        "mlm_epochs": 1,
        "autoregressive_epochs": 2,
        "learning_rate": 0.0008,
        "hidden_dim": 128,
        "num_heads": 4,
        "num_layers": 3,
        "max_eval_candidates": 256,
        "eval_candidate_batch_size": 64,
    },
    "scaled_safe": {
        "max_users": 4096,
        "max_items": 8192,
        "max_review_records": 20_000_000,
        "clip_batch_size": 16,
        "train_batch_size": 32,
        "alignment_batch_size": 256,
        "alignment_epochs": 5,
        "alignment_learning_rate": 0.0005,
        "alignment_hard_negative_count": 16,
        "alignment_window_size": 3,
        "tokenization_batch_size": 256,
        "tokenization_epochs": 15,
        "tokenization_learning_rate": 0.0005,
        "history_max_tokens": 128,
        "mlm_epochs": 5,
        "autoregressive_epochs": 15,
        "learning_rate": 0.0003,
        "hidden_dim": 256,
        "num_heads": 8,
        "num_layers": 4,
        "max_eval_candidates": 1024,
        "eval_candidate_batch_size": 128,
    },
    "long_safe": {
        "max_users": 12288,
        "max_items": 24576,
        "max_review_records": 50_000_000,
        "clip_batch_size": 16,
        "train_batch_size": 32,
        "alignment_batch_size": 256,
        "alignment_epochs": 10,
        "alignment_learning_rate": 0.0004,
        "alignment_hard_negative_count": 24,
        "alignment_window_size": 4,
        "tokenization_batch_size": 256,
        "tokenization_epochs": 30,
        "tokenization_learning_rate": 0.0004,
        "history_max_tokens": 160,
        "mlm_epochs": 10,
        "autoregressive_epochs": 30,
        "learning_rate": 0.00025,
        "hidden_dim": 256,
        "num_heads": 8,
        "num_layers": 4,
        "max_eval_candidates": 1024,
        "eval_candidate_batch_size": 128,
    },
}

KAGGLE_TRAINING_OVERRIDES = dict(KAGGLE_SAFE_PROFILES[KAGGLE_SAFE_PROFILE])

if USE_KAGGLE_SAFE_LIMITS:
    runtime_config = json.loads(json.dumps(FULL_DATASET_CONFIG))
    runtime_config["runtime"]["run_name"] = f"amazon_csj_full_dataset_kaggle_{KAGGLE_SAFE_PROFILE}_train"
    runtime_config["dataset"]["max_users"] = KAGGLE_TRAINING_OVERRIDES["max_users"]
    runtime_config["dataset"]["max_items"] = KAGGLE_TRAINING_OVERRIDES["max_items"]
    runtime_config["dataset"]["max_review_records"] = KAGGLE_TRAINING_OVERRIDES["max_review_records"]
    runtime_config["clip"]["batch_size"] = KAGGLE_TRAINING_OVERRIDES["clip_batch_size"]
    runtime_config["alignment"]["batch_size"] = KAGGLE_TRAINING_OVERRIDES["alignment_batch_size"]
    runtime_config["alignment"]["epochs"] = KAGGLE_TRAINING_OVERRIDES.get("alignment_epochs", runtime_config["alignment"]["epochs"])
    runtime_config["alignment"]["learning_rate"] = KAGGLE_TRAINING_OVERRIDES.get("alignment_learning_rate", runtime_config["alignment"]["learning_rate"])
    runtime_config["alignment"]["hard_negative_count"] = KAGGLE_TRAINING_OVERRIDES.get("alignment_hard_negative_count", runtime_config["alignment"]["hard_negative_count"])
    runtime_config["alignment"]["window_size"] = KAGGLE_TRAINING_OVERRIDES.get("alignment_window_size", runtime_config["alignment"]["window_size"])
    runtime_config["tokenization"]["batch_size"] = KAGGLE_TRAINING_OVERRIDES["tokenization_batch_size"]
    runtime_config["tokenization"]["epochs"] = KAGGLE_TRAINING_OVERRIDES["tokenization_epochs"]
    runtime_config["tokenization"]["learning_rate"] = KAGGLE_TRAINING_OVERRIDES.get("tokenization_learning_rate", runtime_config["tokenization"]["learning_rate"])
    runtime_config["experiment"]["batch_size"] = KAGGLE_TRAINING_OVERRIDES["train_batch_size"]
    runtime_config["experiment"]["history_max_tokens"] = KAGGLE_TRAINING_OVERRIDES["history_max_tokens"]
    runtime_config["experiment"]["mlm_epochs"] = KAGGLE_TRAINING_OVERRIDES["mlm_epochs"]
    runtime_config["experiment"]["autoregressive_epochs"] = KAGGLE_TRAINING_OVERRIDES["autoregressive_epochs"]
    runtime_config["experiment"]["learning_rate"] = KAGGLE_TRAINING_OVERRIDES["learning_rate"]
    runtime_config["experiment"]["hidden_dim"] = KAGGLE_TRAINING_OVERRIDES["hidden_dim"]
    runtime_config["experiment"]["num_heads"] = KAGGLE_TRAINING_OVERRIDES["num_heads"]
    runtime_config["experiment"]["num_layers"] = KAGGLE_TRAINING_OVERRIDES["num_layers"]
    runtime_config["experiment"]["max_eval_candidates"] = KAGGLE_TRAINING_OVERRIDES["max_eval_candidates"]
    runtime_config["experiment"]["eval_candidate_batch_size"] = KAGGLE_TRAINING_OVERRIDES["eval_candidate_batch_size"]
    runtime_config.setdefault("checkpointing", {})["alignment_save_steps"] = KAGGLE_CHECKPOINT_SAVE_STEPS
    runtime_config.setdefault("checkpointing", {})["text_rqvae_save_steps"] = KAGGLE_CHECKPOINT_SAVE_STEPS
    runtime_config.setdefault("checkpointing", {})["vision_rqvae_save_steps"] = KAGGLE_CHECKPOINT_SAVE_STEPS
    runtime_config.setdefault("checkpointing", {})["mlm_save_steps"] = KAGGLE_CHECKPOINT_SAVE_STEPS
    runtime_config.setdefault("checkpointing", {})["autoregressive_save_steps"] = KAGGLE_CHECKPOINT_SAVE_STEPS
    runtime_config.setdefault("checkpointing", {})["restore_from_checkpoints"] = CONTINUE_TRAINING_FROM_CHECKPOINTS

    CONFIG_PATH = STORAGE_ROOT / "configs" / "Full_dataset.kaggle_safe_runtime.json"
    CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with CONFIG_PATH.open("w", encoding="utf-8") as handle:
        json.dump(runtime_config, handle, indent=2)
    FULL_DATASET_CONFIG = runtime_config
    RUN_ROOT = STORAGE_ROOT / "outputs" / runtime_config["runtime"]["run_name"]

DATASET_CONFIG = FULL_DATASET_CONFIG["dataset"]
REVIEWS_RELPATH = DATASET_CONFIG.get("reviews_relpath") or "Clothing_Shoes_and_Jewelry.jsonl"
METADATA_RELPATH = DATASET_CONFIG.get("metadata_relpath") or "meta_Clothing_Shoes_and_Jewelry.jsonl"


def _copy_directory_contents(source_dir: Path, destination_dir: Path) -> None:
    destination_dir.mkdir(parents=True, exist_ok=True)
    for source_child in source_dir.iterdir():
        destination_child = destination_dir / source_child.name
        if source_child.is_dir():
            shutil.copytree(source_child, destination_child, dirs_exist_ok=True)
        else:
            shutil.copy2(source_child, destination_child)


def _looks_like_run_bundle(candidate: Path) -> bool:
    return (
        (candidate / "run_manifest.json").exists()
        or (candidate / "resume_bundle_manifest.json").exists()
        or ((candidate / "checkpoints").exists() and (candidate / "resolved_configs").exists())
    )


def _direct_input_roots() -> list[Path]:
    if not RUNNING_ON_KAGGLE or not KAGGLE_INPUT_ROOT.exists():
        return []
    roots = [path for path in sorted(KAGGLE_INPUT_ROOT.iterdir()) if path.is_dir()]
    nested_root = KAGGLE_INPUT_ROOT / "datasets"
    if nested_root.exists():
        for owner_dir in sorted(path for path in nested_root.iterdir() if path.is_dir()):
            roots.extend(sorted(path for path in owner_dir.iterdir() if path.is_dir()))
    return roots


def _resume_roots_for_slug(slug: str) -> list[Path]:
    roots = [KAGGLE_INPUT_ROOT / slug, KAGGLE_INPUT_ROOT / "datasets" / slug]
    nested_root = KAGGLE_INPUT_ROOT / "datasets"
    if nested_root.exists():
        roots.extend(sorted(nested_root.glob(f"*/{slug}")))
    seen: set[Path] = set()
    deduped = []
    for root in roots:
        if root.exists() and root not in seen:
            deduped.append(root)
            seen.add(root)
    return deduped


def _bundle_layout_candidates(root: Path) -> list[Path]:
    return [
        root,
        root / RUN_ROOT.name,
        root / "outputs" / RUN_ROOT.name,
        root / "masi_artifacts" / "outputs" / RUN_ROOT.name,
        root / "working" / "masi_artifacts" / "outputs" / RUN_ROOT.name,
    ]


def _candidate_resume_bundle_paths() -> list[Path]:
    explicit_roots: list[Path] = []
    for slug in RESUME_BUNDLE_INPUT_SLUGS:
        explicit_roots.extend(_resume_roots_for_slug(str(slug)))
    roots = list(explicit_roots)
    roots.extend(root for root in _direct_input_roots() if root not in roots)

    candidates: list[Path] = []
    for root in roots:
        if root == ATTACHED_DATASET_DIR:
            continue
        if _resolve_extracted_bundle_root(root) is not None:
            candidates.append(root)
        for child in sorted(root.iterdir()):
            if child.is_file() and child.suffix.lower() == ".zip":
                candidates.append(child)
            elif child.is_dir() and child.name != "images" and _resolve_extracted_bundle_root(child) is not None:
                candidates.append(child)
    return candidates


def _resolve_extracted_bundle_root(extracted_root: Path) -> Path | None:
    for candidate in _bundle_layout_candidates(extracted_root):
        if _looks_like_run_bundle(candidate):
            return candidate
    children = [child for child in extracted_root.iterdir() if child.is_dir()]
    if len(children) == 1:
        for candidate in _bundle_layout_candidates(children[0]):
            if _looks_like_run_bundle(candidate):
                return candidate
    return None


def restore_resume_bundle() -> str | None:
    if not (AUTO_RESTORE_RESUME_BUNDLE and RUNNING_ON_KAGGLE):
        return None
    if (RUN_ROOT / "checkpoints").exists() and (RUN_ROOT / "run_manifest.json").exists():
        print(f"Using existing restored run root: {RUN_ROOT}")
        return str(RUN_ROOT)

    restore_workspace = KAGGLE_WORKING_ROOT / "masi_resume_restore"
    restore_workspace.mkdir(parents=True, exist_ok=True)
    for candidate in _candidate_resume_bundle_paths():
        bundle_root: Path | None = None
        if candidate.is_file() and candidate.suffix.lower() == ".zip":
            extract_dir = restore_workspace / candidate.stem
            if extract_dir.exists():
                shutil.rmtree(extract_dir)
            extract_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(candidate, "r") as archive:
                archive.extractall(extract_dir)
            bundle_root = _resolve_extracted_bundle_root(extract_dir)
        elif candidate.is_dir():
            bundle_root = candidate if _looks_like_run_bundle(candidate) else _resolve_extracted_bundle_root(candidate)

        if bundle_root is None:
            continue
        _copy_directory_contents(bundle_root, RUN_ROOT)
        print(f"Restored resume bundle from {candidate} into {RUN_ROOT}")
        return str(candidate)

    print("No prior resume bundle found under /kaggle/input; this run will start from fresh weights.")
    return None


RESTORED_RESUME_BUNDLE = restore_resume_bundle()


def _read_manifest(path: Path) -> dict:
    if not path.exists():
        return {}
    try:
        with path.open("r", encoding="utf-8") as handle:
            return json.load(handle)
    except Exception:
        return {}


def _manifest_chunk_index(payload: dict) -> int | None:
    data_chunk = payload.get("data_chunk")
    if isinstance(data_chunk, dict) and data_chunk.get("chunk_index") is not None:
        return int(data_chunk["chunk_index"])
    launch_dataset = payload.get("launch_dataset")
    if isinstance(launch_dataset, dict) and launch_dataset.get("data_chunk_index") is not None:
        return int(launch_dataset["data_chunk_index"])
    run_manifest = payload.get("run_manifest")
    if isinstance(run_manifest, dict):
        nested = _manifest_chunk_index(run_manifest)
        if nested is not None:
            return nested
    return None


def _previous_chunk_index() -> int | None:
    for manifest_path in [RUN_ROOT / "resume_bundle_manifest.json", RUN_ROOT / "run_manifest.json"]:
        chunk_index = _manifest_chunk_index(_read_manifest(manifest_path))
        if chunk_index is not None:
            return chunk_index
    return None


def _resolve_next_chunk_index() -> int:
    if MANUAL_DATA_CHUNK_INDEX is not None:
        return max(0, int(MANUAL_DATA_CHUNK_INDEX))
    previous = _previous_chunk_index()
    if RESTORED_RESUME_BUNDLE is not None and ADVANCE_DATA_CHUNK_EACH_RUN:
        return (0 if previous is None else previous) + 1
    return 0 if previous is None else previous


DATA_CHUNK_INDEX = _resolve_next_chunk_index()
DATA_CHUNK_USER_RANK_OFFSET = 0
DATA_CHUNK_ITEM_RANK_OFFSET = 0
if DATA_CHUNK_BY == "user_rank":
    DATA_CHUNK_USER_RANK_OFFSET = DATA_CHUNK_INDEX * int(FULL_DATASET_CONFIG["dataset"].get("max_users") or 0)
elif DATA_CHUNK_BY == "item_rank":
    DATA_CHUNK_ITEM_RANK_OFFSET = DATA_CHUNK_INDEX * int(FULL_DATASET_CONFIG["dataset"].get("max_items") or 0)
elif DATA_CHUNK_BY not in {"none", "disabled"}:
    raise ValueError(f"Unsupported DATA_CHUNK_BY: {DATA_CHUNK_BY}")

FULL_DATASET_CONFIG.setdefault("runtime", {})["data_chunk_index"] = DATA_CHUNK_INDEX
FULL_DATASET_CONFIG.setdefault("runtime", {})["data_chunk_by"] = DATA_CHUNK_BY
FULL_DATASET_CONFIG["dataset"]["user_rank_offset"] = DATA_CHUNK_USER_RANK_OFFSET
FULL_DATASET_CONFIG["dataset"]["item_rank_offset"] = DATA_CHUNK_ITEM_RANK_OFFSET
FULL_DATASET_CONFIG["dataset"].setdefault("review_record_offset", 0)
if USE_KAGGLE_SAFE_LIMITS:
    with CONFIG_PATH.open("w", encoding="utf-8") as handle:
        json.dump(FULL_DATASET_CONFIG, handle, indent=2)
DATASET_CONFIG = FULL_DATASET_CONFIG["dataset"]
DATA_CHUNK_STATE = {
    "chunk_index": DATA_CHUNK_INDEX,
    "chunk_by": DATA_CHUNK_BY,
    "user_rank_offset": DATA_CHUNK_USER_RANK_OFFSET,
    "item_rank_offset": DATA_CHUNK_ITEM_RANK_OFFSET,
    "review_record_offset": int(DATASET_CONFIG.get("review_record_offset", 0) or 0),
    "advance_each_restored_run": ADVANCE_DATA_CHUNK_EACH_RUN,
    "manual_chunk_index": MANUAL_DATA_CHUNK_INDEX,
}

print(f"Repository: {REPO_DIR}")
print(f"Repo source: {REPO_URL if RUNNING_ON_KAGGLE and USE_GIT_CLONE_ON_KAGGLE else 'existing checkout'}")
print(f"Source cfg: {SOURCE_CONFIG_PATH}")
print(f"Run config: {CONFIG_PATH}")
print(f"Safe caps:  {USE_KAGGLE_SAFE_LIMITS}")
if USE_KAGGLE_SAFE_LIMITS:
    print(f"Safe profile: {KAGGLE_SAFE_PROFILE}")
    print(json.dumps(KAGGLE_TRAINING_OVERRIDES, indent=2))
print(f"Storage:    {STORAGE_ROOT}")
print(f"Prepared:   {ACTIVE_PREPARED_DIR}")
print(f"Attached:   {ATTACHED_DATASET_DIR}")
print(f"Images:     {ACTIVE_PREPARED_DIR / 'images'}")
print(f"Run root:   {RUN_ROOT}")
print(f"Restored:   {RESTORED_RESUME_BUNDLE}")
print(f"Continue:   {CONTINUE_TRAINING_FROM_CHECKPOINTS}")
print(f"Data chunk: {json.dumps(DATA_CHUNK_STATE)}")

In [ ]:
if RUN_PIP_INSTALL:
    pyproject_path = REPO_DIR / "pyproject.toml"
    pyproject_text = pyproject_path.read_text(encoding="utf-8")
    patched_pyproject_text = pyproject_text.replace('"numpy>=2.4.3"', '"numpy>=1.26,<2.1"')
    if patched_pyproject_text != pyproject_text:
        pyproject_path.write_text(patched_pyproject_text, encoding="utf-8")
        print("Patched pyproject.toml to keep NumPy compatible with Kaggle packages.")

    pip_base = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check"]
    subprocess.run([*pip_base, "--upgrade", "pip"], check=True)
    subprocess.run([*pip_base, "numpy>=1.26,<2.1"], check=True)
    subprocess.run([*pip_base, "-e", ".[recommender]"], check=True)

    import numpy as np

    print(f"Packages ready. NumPy: {np.__version__}")
else:
    print("Skipping package installation.")


In [ ]:
env = dict(os.environ)
env["PYTHONPATH"] = str(REPO_DIR / "src") + os.pathsep + env.get("PYTHONPATH", "")

HF_CACHE_ROOT = STORAGE_ROOT / "hf_cache"
HF_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
hf_env = {
    "HF_HOME": str(HF_CACHE_ROOT),
    "HF_HUB_CACHE": str(HF_CACHE_ROOT / "hub"),
    "TRANSFORMERS_CACHE": str(HF_CACHE_ROOT / "transformers"),
    "HF_HUB_DISABLE_XET": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "TRANSFORMERS_VERBOSITY": "error",
}
os.environ.update(hf_env)
env.update(hf_env)

if RUNNING_ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient

        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
        os.environ["HF_TOKEN"] = hf_token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
        env["HF_TOKEN"] = hf_token
        env["HUGGING_FACE_HUB_TOKEN"] = hf_token
        print("HF_TOKEN loaded from Kaggle secrets.")
    except Exception as exc:
        print(f"HF_TOKEN not configured; continuing unauthenticated. {exc}")
else:
    print("Not running on Kaggle; using existing Hugging Face authentication, if any.")

import torch

if torch.cuda.is_available():
    device = f"cuda:{torch.cuda.current_device()}"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

usage = shutil.disk_usage(REPO_DIR)
print(f"Python: {sys.executable}")
print(f"Torch:  {torch.__version__}")
print(f"Device: {device}")
print(f"Free disk at repo: {usage.free / (1024 ** 3):.1f} GiB")
print(f"HF cache: {HF_CACHE_ROOT}")

assert CONFIG_PATH.exists(), f"Missing config: {CONFIG_PATH}"

In [ ]:
PRELOAD_CLIP_MODEL = True
EXPORT_CLIP_MODEL_BUNDLE = RUNNING_ON_KAGGLE
FORCE_REDOWNLOAD_CLIP_MODEL = False


def looks_like_clip_model_dir(path: Path) -> bool:
    weight_files = ["model.safetensors", "pytorch_model.bin", "tf_model.h5"]
    return (
        path.is_dir()
        and (path / "config.json").exists()
        and (path / "preprocessor_config.json").exists()
        and any((path / name).exists() for name in weight_files)
    )


def find_attached_clip_model_dir() -> Path | None:
    if not RUNNING_ON_KAGGLE or not KAGGLE_INPUT_ROOT.exists():
        return None
    candidates = []
    for base in sorted(KAGGLE_INPUT_ROOT.iterdir()):
        if not base.is_dir():
            continue
        candidates.append(base)
        candidates.extend(path for path in sorted(base.glob("*")) if path.is_dir())
        candidates.extend(path for path in sorted(base.glob("*/*")) if path.is_dir())
        candidates.extend(path for path in sorted(base.glob("*/*/*")) if path.is_dir())
    for candidate in candidates:
        if looks_like_clip_model_dir(candidate):
            return candidate
    return None


if PRELOAD_CLIP_MODEL:
    import time
    from transformers import CLIPModel, CLIPProcessor

    clip_model_name = FULL_DATASET_CONFIG.get("clip", {}).get("model_name", "openai/clip-vit-base-patch32")
    attached_clip_dir = find_attached_clip_model_dir()
    local_clip_dir = attached_clip_dir or (STORAGE_ROOT / "hf_models" / clip_model_name.replace("/", "_"))
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN") or None
    started_at = time.time()

    if attached_clip_dir is not None:
        print(f"Using attached CLIP model directory: {attached_clip_dir}")
    elif looks_like_clip_model_dir(local_clip_dir) and not FORCE_REDOWNLOAD_CLIP_MODEL:
        print(f"Using existing local CLIP model directory: {local_clip_dir}")
    else:
        print(f"Downloading CLIP once into reusable directory: {local_clip_dir}")
        local_clip_dir.mkdir(parents=True, exist_ok=True)
        processor = CLIPProcessor.from_pretrained(clip_model_name, token=hf_token)
        model = CLIPModel.from_pretrained(
            clip_model_name,
            token=hf_token,
            low_cpu_mem_usage=False,
            use_safetensors=True,
        )
        processor.save_pretrained(local_clip_dir)
        model.save_pretrained(local_clip_dir, safe_serialization=True)
        del model, processor

    os.environ["MASI_CLIP_MODEL_DIR"] = str(local_clip_dir)
    env["MASI_CLIP_MODEL_DIR"] = str(local_clip_dir)
    FULL_DATASET_CONFIG.setdefault("clip", {})["local_model_path"] = str(local_clip_dir)

    if RUNNING_ON_KAGGLE or CONFIG_PATH != SOURCE_CONFIG_PATH:
        if CONFIG_PATH == SOURCE_CONFIG_PATH:
            CONFIG_PATH = STORAGE_ROOT / "configs" / "Full_dataset.kaggle_runtime.json"
        CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
        with CONFIG_PATH.open("w", encoding="utf-8") as handle:
            json.dump(FULL_DATASET_CONFIG, handle, indent=2)

    clip_archive_path = None
    if EXPORT_CLIP_MODEL_BUNDLE and attached_clip_dir is None:
        archive_base = local_clip_dir
        clip_archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=local_clip_dir)

    print(json.dumps({
        "clip_model_source": str(local_clip_dir),
        "clip_model_archive_for_kaggle_upload": clip_archive_path,
        "run_config": str(CONFIG_PATH),
        "elapsed_minutes": round((time.time() - started_at) / 60, 2),
    }, indent=2))
else:
    print("Skipping CLIP preload/export.")


## Prepared Kaggle Dataset

The Kaggle dataset is already prepared and includes reviews, metadata, images, and manifests. The notebook validates those files before training.

In [ ]:
required_dataset_paths = [ACTIVE_PREPARED_DIR / entry for entry in REQUIRED_DATASET_ENTRIES]
missing_dataset_paths = [path for path in required_dataset_paths if not path.exists()]

if missing_dataset_paths:
    raise FileNotFoundError(
        "Missing prepared dataset entries:\n"
        + "\n".join(f"- {path}" for path in missing_dataset_paths)
        + "\nExpected Kaggle dataset path: "
        + str(KAGGLE_FULL_DATASET_DIR)
    )
else:
    print("Prepared full_dataset input is complete.")
    for path in required_dataset_paths:
        print(f"- {path}")

## Prepare `data/full_dataset`

The `full_dataset` preset scans up to 20,000,000 review records, applies 5-core filtering, caps the selected subset at 102,400 users and 204,800 items, and writes a disk-backed SQLite selection index for lineage.

In [ ]:
prepared_reviews = ACTIVE_PREPARED_DIR / REVIEWS_RELPATH
prepared_metadata = ACTIVE_PREPARED_DIR / METADATA_RELPATH
subset_manifest = ACTIVE_PREPARED_DIR / "subset_manifest.json"

need_prepare = (not USING_ATTACHED_PREPARED_DATASET) and (
    FORCE_PREPARE or not (prepared_reviews.exists() and prepared_metadata.exists() and subset_manifest.exists())
)

if USING_ATTACHED_PREPARED_DATASET:
    print(f"Prepared full_dataset is attached read-only at {ACTIVE_PREPARED_DIR}.")
elif need_prepare:
    PREPARED_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            sys.executable,
            "scripts/prepare_amazon_csj_subset.py",
            "--reviews-path",
            str(RAW_REVIEWS_PATH),
            "--metadata-path",
            str(RAW_METADATA_PATH),
            "--output-dir",
            str(PREPARED_DIR),
            "--preset",
            "full_dataset",
        ],
        check=True,
        env=env,
    )
else:
    print("Prepared full_dataset files already exist; set FORCE_PREPARE = True to rebuild.")

if subset_manifest.exists():
    with subset_manifest.open("r", encoding="utf-8") as handle:
        subset_summary = json.load(handle)
else:
    subset_summary = {"note": "subset_manifest.json was not found; training can still use the configured prepared JSONL files."}

print(json.dumps({
    "selected_user_count": subset_summary.get("selected_user_count"),
    "selected_item_count": subset_summary.get("selected_item_count"),
    "selected_interaction_count": subset_summary.get("selected_interaction_count"),
    "subset_manifest": str(subset_manifest),
}, indent=2))

## Download And Validate Images

This writes validated images to `data/full_dataset/images` and records image coverage in `image_download_manifest.json`.

In [ ]:
image_manifest = ACTIVE_PREPARED_DIR / "image_download_manifest.json"
configured_image_manifest = RUN_ROOT / "image_download_manifest.json"

if PREFETCH_IMAGES and USING_ATTACHED_PREPARED_DATASET:
    subprocess.run(
        [
            sys.executable,
            "scripts/download_amazon_csj_images.py",
            "--config",
            str(CONFIG_PATH),
            "--storage-root",
            str(STORAGE_ROOT),
            "--workers",
            str(IMAGE_WORKERS),
            "--retries",
            "2",
            "--timeout-seconds",
            "30",
            "--resume",
        ],
        check=True,
        env=env,
    )
elif PREFETCH_IMAGES:
    subprocess.run(
        [
            sys.executable,
            "scripts/download_amazon_csj_subset_images.py",
            "--metadata-path",
            str(prepared_metadata),
            "--output-dir",
            str(PREPARED_DIR),
            "--workers",
            str(IMAGE_WORKERS),
            "--retries",
            "2",
            "--timeout-seconds",
            "30",
            "--resume",
        ],
        check=True,
        env=env,
    )
else:
    print("Skipping image prefetch. train_masi.py can still download missing images if configured.")

summary_manifest = configured_image_manifest if configured_image_manifest.exists() else image_manifest
if summary_manifest.exists():
    with summary_manifest.open("r", encoding="utf-8") as handle:
        image_summary = json.load(handle)
    print(json.dumps({
        "selected_item_count": image_summary.get("selected_item_count"),
        "successful_item_count": image_summary.get("successful_item_count"),
        "failed_item_count": image_summary.get("failed_item_count"),
        "missing_url_item_count": image_summary.get("missing_url_item_count"),
        "image_manifest": str(summary_manifest),
    }, indent=2))

## Run MASI Training Stages

The cells below keep the existing repository script boundaries to avoid risky rewrites, but they no longer hide the whole run behind one monolithic notebook cell.

Execution cells are split into durable handoff points:

1. resolve and persist stage configs,
2. build Phase 1/2 MASI tokens,
3. inspect Phase 1 alignment, text RQ-VAE, and vision RQ-VAE artifacts,
4. run Phase 3 with capped, batched evaluation candidates and visible evaluation progress,
5. inspect MLM, autoregressive, warm metric, cold metric, final summary, and checkpoint outputs.

`run_masi_experiment.py` writes final MLM/autoregressive checkpoints before ranking evaluation starts, so a long evaluation cannot erase the value of completed training.


In [ ]:
from importlib import util as importlib_util
from typing import Any


def apply_long_prefix_decode_hotfix() -> None:
    generative_path = REPO_DIR / "src" / "masi" / "recommender" / "generative.py"
    source = generative_path.read_text(encoding="utf-8")
    old = """        generated = prefix_token_ids.clone()\n        for _ in range(max_new_tokens):\n            logits = self.forward(generated)\n            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)\n"""
    new = """        generated = prefix_token_ids.clone()\n        for _ in range(max_new_tokens):\n            # Long user histories can exceed the fixed positional-embedding\n            # budget used during training. Decode from the most recent context,\n            # matching the left-truncation policy used by ranking/evaluation.\n            model_input = generated[:, -self.max_sequence_length :]\n            logits = self.forward(model_input)\n            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)\n"""
    if old in source:
        generative_path.write_text(source.replace(old, new), encoding="utf-8")
        print(f"Applied long-prefix decode hotfix: {generative_path}")
    elif "model_input = generated[:, -self.max_sequence_length :]" in source:
        print("Long-prefix decode hotfix already present.")
    else:
        raise RuntimeError(f"Could not apply long-prefix decode hotfix to {generative_path}")


def load_script_module(module_name: str, script_path: Path):
    spec = importlib_util.spec_from_file_location(module_name, script_path)
    module = importlib_util.module_from_spec(spec)
    assert spec is not None and spec.loader is not None
    spec.loader.exec_module(module)
    return module


def maybe_display_json(title: str, payload: dict[str, Any]) -> None:
    try:
        from IPython.display import JSON, Markdown, display

        display(Markdown(f"### {title}"))
        display(JSON(payload, expanded=False))
    except Exception:
        print(f"\n### {title}")
        print(json.dumps(payload, indent=2))


def read_json_if_exists(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def run_stage_command(stage_name: str, command: list[str], expected_paths: list[Path]) -> None:
    missing = [path for path in expected_paths if not path.exists()]
    if missing or FORCE_TRAIN or CONTINUE_TRAINING_FROM_CHECKPOINTS:
        print(f"Running {stage_name}...")
        print(" ".join(str(part) for part in command))
        subprocess.run(command, check=True, env=env, cwd=REPO_DIR)
    else:
        print(f"Skipping {stage_name}; expected artifacts already exist. Set FORCE_TRAIN = True or CONTINUE_TRAINING_FROM_CHECKPOINTS = True to rerun.")
    for expected_path in expected_paths:
        assert expected_path.exists(), f"Expected artifact missing after {stage_name}: {expected_path}"


def latest_checkpoint_payload(stage_dir: Path) -> dict[str, Any]:
    latest_path = stage_dir / "latest.json"
    if not latest_path.exists():
        return {"latest_manifest": str(latest_path), "status": "missing"}
    payload = read_json_if_exists(latest_path)
    payload["latest_manifest"] = str(latest_path)
    return payload


apply_long_prefix_decode_hotfix()


In [ ]:
sys.path.insert(0, str(REPO_DIR / "src"))
from masi.common.io import ensure_directory, write_json
from masi.common.runtime import detect_runtime_environment, resolve_input_path, resolve_path

train_launcher = load_script_module("masi_train_launcher", REPO_DIR / "scripts" / "train_masi.py")
config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
runtime_config = dict(config.get("runtime", {}))
dataset_config = dict(config["dataset"])
assets_config = dict(config.get("assets", {}))
environment = detect_runtime_environment()
run_name = str(runtime_config.get("run_name", "masi_train_csj"))
RUN_ROOT = ensure_directory(STORAGE_ROOT / "outputs" / run_name)
resolved_config_root = ensure_directory(RUN_ROOT / "resolved_configs")
checkpoint_root = ensure_directory(RUN_ROOT / "checkpoints")

resolved_dataset_root = ATTACHED_DATASET_DIR
reviews_path = resolve_input_path(
    repo_root=REPO_DIR,
    storage_root=STORAGE_ROOT,
    configured_path=str(dataset_config.get("reviews_path", "")),
    kaggle_dataset_root=resolved_dataset_root,
    relative_path=str(dataset_config.get("reviews_relpath", "")).strip() or None,
)
metadata_path = resolve_input_path(
    repo_root=REPO_DIR,
    storage_root=STORAGE_ROOT,
    configured_path=str(dataset_config.get("metadata_path", "")),
    kaggle_dataset_root=resolved_dataset_root,
    relative_path=str(dataset_config.get("metadata_relpath", "")).strip() or None,
)
assert reviews_path is not None
assert metadata_path is not None
if resolved_dataset_root is None and reviews_path.exists() and metadata_path.exists() and reviews_path.parent == metadata_path.parent:
    resolved_dataset_root = reviews_path.parent

data_setup = train_launcher._ensure_dataset_inputs(
    repo_root=REPO_DIR,
    config=config,
    reviews_path=reviews_path,
    metadata_path=metadata_path,
    dataset_root=resolved_dataset_root,
)

token_outputs_root = ensure_directory(RUN_ROOT / "phase12_tokens")
experiment_outputs_root = ensure_directory(RUN_ROOT / "phase3_experiment")
image_cache_dir = resolve_path(
    STORAGE_ROOT,
    str(assets_config.get("image_cache_dir", f"data/processed/{run_name}/images")),
)
metadata_cache_path = resolve_path(
    STORAGE_ROOT,
    str(assets_config.get("metadata_cache_path", f"data/processed/{run_name}/metadata.slice.jsonl")),
)
assert image_cache_dir is not None
assert metadata_cache_path is not None

preloaded_image_dir = resolve_input_path(
    repo_root=REPO_DIR,
    storage_root=STORAGE_ROOT,
    configured_path=str(assets_config.get("preloaded_images_path", "")).strip() or None,
    kaggle_dataset_root=resolved_dataset_root,
    relative_path=str(assets_config.get("preloaded_images_relpath", "")).strip() or None,
)
preloaded_image_dirs = [
    str(preloaded_image_dir.resolve())
] if preloaded_image_dir is not None and preloaded_image_dir.exists() else []

token_config = {
    "seed": int(config["seed"]),
    "dataset": {
        "reviews_path": str(reviews_path.resolve()),
        "min_user_interactions": int(dataset_config["min_user_interactions"]),
        "min_item_interactions": int(dataset_config.get("min_item_interactions", dataset_config["min_user_interactions"])),
        "max_users": dataset_config.get("max_users"),
        "max_items": dataset_config.get("max_items"),
        "max_review_records": dataset_config.get("max_review_records"),
        "review_record_offset": dataset_config.get("review_record_offset", 0),
        "user_rank_offset": dataset_config.get("user_rank_offset", 0),
        "item_rank_offset": dataset_config.get("item_rank_offset", 0),
        "collapse_consecutive_duplicates": bool(dataset_config.get("collapse_consecutive_duplicates", False)),
    },
    "assets": {
        "metadata_local_path": str(metadata_path.resolve()),
        "use_remote_metadata": bool(assets_config.get("use_remote_metadata", False)),
        "metadata_cache_path": str(metadata_cache_path.resolve()),
        "image_cache_dir": str(image_cache_dir.resolve()),
        "preloaded_image_dirs": preloaded_image_dirs,
        "download_missing_images": bool(assets_config.get("download_missing_images", True)),
        "image_download_workers": int(assets_config.get("image_download_workers", 1)),
        "image_download_retries": int(assets_config.get("image_download_retries", 0)),
        "image_download_timeout_seconds": int(assets_config.get("image_download_timeout_seconds", 30)),
        "image_download_resume": bool(assets_config.get("image_download_resume", True)),
    },
    "clip": dict(config["clip"]),
    "alignment": dict(config["alignment"]),
    "tokenization": dict(config["tokenization"]),
    "checkpointing": dict(config.get("checkpointing", {})),
    "method_toggles": dict(config.get("method_toggles", {})),
    "outputs_root": str(token_outputs_root.resolve()),
    "fused_ids_path": str((token_outputs_root / "fused_semantic_ids.jsonl").resolve()),
    "checkpoint_root": str((checkpoint_root / "phase12_tokens").resolve()),
}
token_config_path = write_json(token_config, resolved_config_root / "token_build.json")
launch_config_path = write_json(config, resolved_config_root / "launch_config.json")

experiment_config = {
    "seed": int(config["seed"]),
    "history_max_tokens": int(config["experiment"]["history_max_tokens"]),
    "target_max_tokens": config["experiment"].get("target_max_tokens"),
    "mlm_max_tokens": config["experiment"].get("mlm_max_tokens"),
    "batch_size": int(config["experiment"]["batch_size"]),
    "learning_rate": float(config["experiment"]["learning_rate"]),
    "hidden_dim": int(config["experiment"]["hidden_dim"]),
    "num_heads": int(config["experiment"]["num_heads"]),
    "num_layers": int(config["experiment"]["num_layers"]),
    "dropout": float(config["experiment"]["dropout"]),
    "mlm_epochs": int(config["experiment"]["mlm_epochs"]),
    "autoregressive_epochs": int(config["experiment"]["autoregressive_epochs"]),
    "top_k": int(config["experiment"]["top_k"]),
    "cold_start_ratio": float(config["experiment"]["cold_start_ratio"]),
    "min_train_history": int(config["experiment"]["min_train_history"]),
    "min_sequence_items": int(config["experiment"]["min_sequence_items"]),
    "max_eval_candidates": config["experiment"].get("max_eval_candidates"),
    "eval_candidate_batch_size": config["experiment"].get("eval_candidate_batch_size"),
    "outputs_root": str(experiment_outputs_root.resolve()),
    "checkpoint_root": str((checkpoint_root / "phase3_experiment").resolve()),
    "token_artifact_path": token_config["fused_ids_path"],
    "require_token_artifact": True,
    "checkpointing": dict(config.get("checkpointing", {})),
    "method_toggles": dict(config.get("method_toggles", {})),
    "dataset": token_config["dataset"],
}
experiment_config_path = write_json(experiment_config, resolved_config_root / "experiment.json")

run_manifest_path = RUN_ROOT / "run_manifest.json"
token_summary_path = token_outputs_root / "masi_token_summary.json"
experiment_summary_path = experiment_outputs_root / "experiment_summary.json"

maybe_display_json("Resolved Stage Plan", {
    "run_root": str(RUN_ROOT),
    "launch_config": str(launch_config_path),
    "token_config": str(token_config_path),
    "experiment_config": str(experiment_config_path),
    "token_outputs_root": str(token_outputs_root),
    "experiment_outputs_root": str(experiment_outputs_root),
    "checkpoint_root": str(checkpoint_root),
    "preloaded_image_dirs": preloaded_image_dirs,
    "restored_resume_bundle": RESTORED_RESUME_BUNDLE,
    "continue_training_from_checkpoints": CONTINUE_TRAINING_FROM_CHECKPOINTS,
    "data_chunk": DATA_CHUNK_STATE,
    "evaluation_controls": {
        "max_eval_candidates": experiment_config.get("max_eval_candidates"),
        "eval_candidate_batch_size": experiment_config.get("eval_candidate_batch_size"),
    },
})


## Build MASI Tokens

This cell runs the existing Phase 1/2 token builder. Internally it trains behavior-aware alignment, then the text and vision RQ-VAE codebooks, and writes checkpoints as each model finishes.


In [ ]:
run_stage_command(
    "Phase 1/2 MASI token build",
    [sys.executable, "scripts/build_masi_tokens.py", "--config", str(token_config_path)],
    [token_summary_path, Path(token_config["fused_ids_path"])],
)

token_summary = read_json_if_exists(token_summary_path)
maybe_display_json("MASI Token Build Summary", {
    "summary_path": str(token_summary_path),
    "items_with_full_modalities": token_summary.get("items_with_full_modalities"),
    "users_after_modality_filter": token_summary.get("users_after_modality_filter"),
    "fused_ids_path": token_summary.get("fused_ids_path"),
    "checkpoint_paths": token_summary.get("checkpoint_paths", {}),
})


## Phase 1 Alignment Output

In [ ]:
token_summary = read_json_if_exists(token_summary_path)
maybe_display_json("Phase 1 Behavior Alignment", {
    "status": token_summary.get("alignment_status"),
    "positive_pairs": token_summary.get("positive_pairs"),
    "alignment_steps": token_summary.get("alignment_steps"),
    "alignment_last_loss": token_summary.get("alignment_last_loss"),
    "final_checkpoint": token_summary.get("checkpoint_paths", {}).get("behavior_alignment"),
    "latest_periodic_checkpoint": token_summary.get("periodic_latest_checkpoint_paths", {}).get("behavior_alignment"),
    "latest_manifest": latest_checkpoint_payload(checkpoint_root / "phase12_tokens" / "behavior_alignment_steps"),
})


## Text RQ-VAE Output

In [ ]:
token_summary = read_json_if_exists(token_summary_path)
maybe_display_json("Text RQ-VAE", {
    "last_loss": token_summary.get("text_quantization_last_loss"),
    "unique_text_code_sequences": token_summary.get("unique_text_code_sequences"),
    "final_checkpoint": token_summary.get("checkpoint_paths", {}).get("text_rqvae"),
    "latest_periodic_checkpoint": token_summary.get("periodic_latest_checkpoint_paths", {}).get("text_rqvae"),
    "latest_manifest": latest_checkpoint_payload(checkpoint_root / "phase12_tokens" / "text_rqvae_steps"),
})


## Vision RQ-VAE Output

In [ ]:
token_summary = read_json_if_exists(token_summary_path)
maybe_display_json("Vision RQ-VAE", {
    "last_loss": token_summary.get("image_quantization_last_loss"),
    "unique_visual_code_sequences": token_summary.get("unique_visual_code_sequences"),
    "final_checkpoint": token_summary.get("checkpoint_paths", {}).get("vision_rqvae"),
    "latest_periodic_checkpoint": token_summary.get("periodic_latest_checkpoint_paths", {}).get("vision_rqvae"),
    "latest_manifest": latest_checkpoint_payload(checkpoint_root / "phase12_tokens" / "vision_rqvae_steps"),
})


## Run Phase 3 Experiment

This cell trains cross-modal MLM, trains the autoregressive recommender, saves final checkpoints, then evaluates warm and cold splits with progress bars. Candidate ranking is capped and scored in batches according to the resolved experiment config.


In [ ]:
run_stage_command(
    "Phase 3 MLM + autoregressive + warm/cold evaluation",
    [sys.executable, "scripts/run_masi_experiment.py", "--config", str(experiment_config_path)],
    [experiment_summary_path],
)

experiment_summary = read_json_if_exists(experiment_summary_path)
maybe_display_json("Phase 3 Experiment Summary", {
    "summary_path": str(experiment_summary_path),
    "mlm_status": experiment_summary.get("mlm_status"),
    "generative_finetuning_status": experiment_summary.get("generative_finetuning_status"),
    "vocab_size": experiment_summary.get("vocab_size"),
    "num_items": experiment_summary.get("num_items"),
    "num_train_examples": experiment_summary.get("num_train_examples"),
    "num_mlm_examples": experiment_summary.get("num_mlm_examples"),
    "checkpoint_paths": experiment_summary.get("checkpoint_paths", {}),
})


## MLM Output

In [ ]:
experiment_summary = read_json_if_exists(experiment_summary_path)
maybe_display_json("Cross-Modal MLM", {
    "status": experiment_summary.get("mlm_status"),
    "loss_history": experiment_summary.get("mlm_loss_history", []),
    "final_checkpoint": experiment_summary.get("checkpoint_paths", {}).get("cross_modal_mlm"),
    "latest_periodic_checkpoint": experiment_summary.get("periodic_latest_checkpoint_paths", {}).get("cross_modal_mlm"),
    "training_artifact_summary": experiment_summary.get("checkpoint_paths", {}).get("training_artifact_summary"),
})


## Autoregressive Recommender Output

In [ ]:
experiment_summary = read_json_if_exists(experiment_summary_path)
maybe_display_json("Autoregressive Recommender", {
    "status": experiment_summary.get("generative_finetuning_status"),
    "loss_history": experiment_summary.get("autoregressive_loss_history", []),
    "final_checkpoint": experiment_summary.get("checkpoint_paths", {}).get("generative_recommender"),
    "latest_periodic_checkpoint": experiment_summary.get("periodic_latest_checkpoint_paths", {}).get("generative_recommender"),
    "sample_generation": experiment_summary.get("sample_generation", {}),
})


## Warm Metrics

In [ ]:
experiment_summary = read_json_if_exists(experiment_summary_path)
maybe_display_json("Warm-Start Ranking Metrics", experiment_summary.get("warm_metrics", {}))


## Cold Metrics

In [ ]:
experiment_summary = read_json_if_exists(experiment_summary_path)
maybe_display_json("Zero-Shot Cold-Start Ranking Metrics", experiment_summary.get("cold_metrics", {}))


## Save Final Summary And Checkpoints

In [ ]:
for required_path in [token_summary_path, experiment_summary_path]:
    assert required_path.exists(), f"Expected artifact does not exist: {required_path}"

token_summary = read_json_if_exists(token_summary_path)
experiment_summary = read_json_if_exists(experiment_summary_path)
manifest = {
    "environment": environment,
    "storage_root": str(STORAGE_ROOT.resolve()),
    "run_root": str(RUN_ROOT.resolve()),
    "resolved_config_paths": {
        "token_build": str(token_config_path.resolve()),
        "experiment": str(experiment_config_path.resolve()),
        "launch_config": str(launch_config_path.resolve()),
    },
    "data_setup": data_setup,
    "resolved_dataset_root": str(resolved_dataset_root.resolve()) if resolved_dataset_root is not None else None,
    "preloaded_image_dirs": preloaded_image_dirs,
    "restored_resume_bundle": RESTORED_RESUME_BUNDLE,
    "continue_training_from_checkpoints": CONTINUE_TRAINING_FROM_CHECKPOINTS,
    "data_chunk": DATA_CHUNK_STATE,
    "token_summary_path": str(token_summary_path.resolve()),
    "experiment_summary_path": str(experiment_summary_path.resolve()),
    "token_summary": token_summary,
    "experiment_summary": experiment_summary,
}
with run_manifest_path.open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

maybe_display_json("Final Run Summary", {
    "run_manifest": str(run_manifest_path),
    "fused_ids_path": token_summary.get("fused_ids_path"),
    "items_with_full_modalities": token_summary.get("items_with_full_modalities"),
    "warm_metrics": experiment_summary.get("warm_metrics", {}),
    "cold_metrics": experiment_summary.get("cold_metrics", {}),
    "token_checkpoints": token_summary.get("checkpoint_paths", {}),
    "phase3_checkpoints": experiment_summary.get("checkpoint_paths", {}),
})


## Checkpoint Inventory

In [ ]:
checkpoint_root = RUN_ROOT / "checkpoints"
interesting_paths = [
    RUN_ROOT / "resolved_configs" / "token_build.json",
    RUN_ROOT / "resolved_configs" / "experiment.json",
    RUN_ROOT / "phase12_tokens" / "fused_semantic_ids.jsonl",
    checkpoint_root / "phase12_tokens" / "behavior_alignment.pt",
    checkpoint_root / "phase12_tokens" / "text_rqvae.pt",
    checkpoint_root / "phase12_tokens" / "vision_rqvae.pt",
    checkpoint_root / "phase3_experiment" / "cross_modal_mlm.pt",
    checkpoint_root / "phase3_experiment" / "generative_recommender.pt",
    checkpoint_root / "phase3_experiment" / "training_artifact_summary.json",
    checkpoint_root,
]

inventory = {str(path): ("exists" if path.exists() else "missing") for path in interesting_paths}
latest_manifests = sorted(checkpoint_root.rglob("latest.json")) if checkpoint_root.exists() else []
latest = {}
for manifest_path in latest_manifests:
    payload = read_json_if_exists(manifest_path)
    latest[str(manifest_path)] = {
        "global_step": payload.get("global_step"),
        "checkpoint_path": payload.get("checkpoint_path"),
    }
maybe_display_json("Checkpoint Inventory", {"paths": inventory, "latest_periodic_checkpoints": latest})


## Export Run Outputs

The export bundle includes the selected `RUN_ROOT` outputs and checkpoints. It does not include raw files or the attached read-only image dataset.

Kaggle working storage is volatile. After this cell creates the zip, use Kaggle `Save Version` or publish the zip as a private Dataset before restarting the session.


In [ ]:
if EXPORT_BUNDLE:
    checkpoint_root = RUN_ROOT / "checkpoints"
    checkpoint_files = [
        str(path.relative_to(RUN_ROOT))
        for path in sorted(checkpoint_root.rglob("*"))
        if path.is_file()
    ] if checkpoint_root.exists() else []
    bundle_manifest_path = RUN_ROOT / "resume_bundle_manifest.json"
    with bundle_manifest_path.open("w", encoding="utf-8") as handle:
        json.dump(
            {
                "run_root_name": RUN_ROOT.name,
                "restored_resume_bundle": RESTORED_RESUME_BUNDLE,
                "continue_training_from_checkpoints": CONTINUE_TRAINING_FROM_CHECKPOINTS,
                "data_chunk": DATA_CHUNK_STATE,
                "checkpoint_files": checkpoint_files,
                "required_next_restore_root": str(RUN_ROOT),
                "notes": "Attach this zip or an unzipped copy as a Kaggle Dataset before the next run; the notebook auto-restores it into RUN_ROOT.",
            },
            handle,
            indent=2,
        )
    archive_base = STORAGE_ROOT / "outputs" / f"{RUN_ROOT.name}_artifacts"
    zip_path = shutil.make_archive(str(archive_base), "zip", root_dir=RUN_ROOT)
    print(f"Exported resume-ready run bundle: {zip_path}")
    print(f"Bundle manifest: {bundle_manifest_path}")
else:
    print("Skipping export bundle.")
